# Federated Runtime: HuggingFace Fine-Tuning

In this notebook, you will learn how to fine-tune a text-classification model on the IMDb dataset using Federated Learning.

We will begin by simulating the entire workflow locally (using `LocalRuntime`), then deploy that same workflow into a Federated infrastructure (using `FederatedRuntime`).


**Note:**

Cells marked with the `#| export` directive will be automatically exported to the FL workspace. This workspace is then shared with all Federated Learning clients for execution.

The export directive is only required when using the `FederatedRuntime`.

# Getting Started

In the following cell `#| default_exp` experiment directive sets the name of the python module as `experiment`. This name can be customized according to the user’s requirements and preferences.

In [ ]:
# | default_exp experiment

### Installing requirements

We begin with installing the required packages and dependencies

In [ ]:
# | export

%pip install git+https://github.com/securefederatedai/openfl.git
%pip install -r ../../../workflow_interface_requirements.txt
%pip install -U datasets==3.0.0
%pip install -U transformers==4.44.2
%pip install -U evaluate==0.4.3
%pip install -U ipywidgets
%pip install -U torch==2.4.1
%pip install -U accelerate==0.34.2
%pip install -U termcolor

### Defining global variables and functions

Next, we define hyperparameters and helper functions that are required throughout this tutorial

In [ ]:
# | export

import evaluate
import numpy as np
from transformers import AutoTokenizer

# Hyperparameters
RANDOM_SEED = 12345
MODEL_NAME = "prajjwal1/bert-tiny"
NUM_LABELS = 2
MAX_MODEL_LEN = 512
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PER_DEVICE_BATCH = 512
AUTO_FIND_BATCH_SIZE = True
NUM_TRAIN_EPOCHS = 3
FL_ROUNDS = 2

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_MODEL_LEN,
    )


# Accuracy metrics
def compute_metrics(eval_pred):
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")

    preds = np.argmax(eval_pred.predictions, axis=1)

    acc = accuracy_metric.compute(
        predictions=preds,
        references=eval_pred.label_ids,
    )["accuracy"]
    f1 = f1_metric.compute(
        predictions=preds,
        references=eval_pred.label_ids,
        average="weighted",
    )["f1"]

    return {"accuracy": acc, "f1": f1}

### Defining Federated Averaging function

Next, we define a helper function for averaging the weights of models

In [ ]:
# | export

import torch


def fed_avg(agg_model, client_models, weights=None):
    client_state_dicts = [m.state_dict() for m in client_models]
    agg_state_dict = agg_model.state_dict()
    device = next(agg_model.parameters()).device
    dtype = next(agg_model.parameters()).dtype

    if weights is None:
        num_models = len(client_models)
        weights = torch.ones(num_models, dtype=dtype, device=device) / num_models
    else:
        weights = torch.tensor(weights, dtype=dtype, device=device)

    with torch.no_grad():
        for key in agg_state_dict:
            stacked_tensors = torch.stack(
                [sd[key].to(device) for sd in client_state_dicts],
                dim=0,
            )

            w = weights.view(-1, *[1] * (stacked_tensors.dim() - 1))
            avg_tensor = torch.sum(stacked_tensors * w, dim=0)

            agg_state_dict[key] = avg_tensor

    agg_model.load_state_dict(agg_state_dict)
    return agg_model

### Defining the federated workflow

Next, we define the federated workflow that contains validation and training processes

- FLSpec – Defines the flow specification. User defined flows are subclasses of this.
- aggregator/collaborator - placement decorators that define where the task will be assigned

In [ ]:
# | export

from termcolor import colored
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed

from openfl.experimental.workflow.interface import FLSpec
from openfl.experimental.workflow.placement import aggregator, collaborator


class FederatedFlowHF(FLSpec):
    """
    This Flow fine-tunes a text classification model from HuggingFace.
    """

    def __init__(self, model=None, **kwargs):
        super().__init__(**kwargs)

        if model is not None:
            self.model = model
        else:
            set_seed(RANDOM_SEED)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=NUM_LABELS,
            )

        self.rounds = FL_ROUNDS
        self.results = []

    @aggregator
    def start(self):
        """
        This is the start of the Flow.
        """
        tag = colored("[Aggregator]", "white", "on_magenta")
        print(tag, "Initializing Workflow ...")

        self.collaborators = self.runtime.collaborators
        self.current_round = 0

        self.next(self.aggregated_model_validation, foreach="collaborators")

    @collaborator
    def aggregated_model_validation(self):
        """
        Perform validation of aggregated model on collaborators.
        """
        tag = colored(f"[Collab: {self.input}]", "white", "on_blue")
        print(tag, "Performing Validation on aggregated model ...")

        test_ds = self.test_dataset
        tokenized_test = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])

        eval_args = TrainingArguments(
            output_dir="trainer_output",
            per_device_eval_batch_size=PER_DEVICE_BATCH,
            auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
            do_train=False,
            do_eval=True,
            logging_strategy="no",
            save_strategy="no",
            report_to=[],
        )
        trainer = Trainer(
            model=self.model,
            args=eval_args,
            eval_dataset=tokenized_test,
            compute_metrics=compute_metrics,
        )

        eval_metrics = trainer.evaluate()
        self.agg_validation_accuracy = eval_metrics["eval_accuracy"]
        self.agg_validation_f1 = eval_metrics["eval_f1"]
        print(
            tag,
            f"Aggregated Model validation accuracy = {self.agg_validation_accuracy:.4f}",
            f"F1 = {self.agg_validation_f1:.4f}",
        )

        self.next(self.train)

    @collaborator
    def train(self):
        """
        Train model on Local collaborator dataset.
        """
        tag = colored(f"[Collab: {self.input}]", "white", "on_blue")
        print(tag, "Training Model on local dataset ...")

        train_ds = self.train_dataset
        tokenized_train = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])

        train_args = TrainingArguments(
            output_dir="trainer_output",
            eval_strategy="no",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=PER_DEVICE_BATCH,
            auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            weight_decay=WEIGHT_DECAY,
            logging_strategy="no",
            save_strategy="no",
            report_to=[],
        )

        trainer = Trainer(
            model=self.model,
            args=train_args,
            train_dataset=tokenized_train,
        )

        train_output = trainer.train()
        self.loss = train_output.training_loss
        print(tag, f"Local training loss = {self.loss:.4f}")

        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        """
        Validate locally trained model.
        """
        tag = colored(f"[Collab: {self.input}]", "white", "on_blue")
        print(tag, "Performing Validation on locally trained model ...")

        test_ds = self.test_dataset
        tokenized_test = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])

        eval_args = TrainingArguments(
            output_dir="trainer_output",
            per_device_eval_batch_size=PER_DEVICE_BATCH,
            auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
            do_train=False,
            do_eval=True,
            logging_strategy="no",
            save_strategy="no",
            report_to=[],
        )
        trainer = Trainer(
            model=self.model,
            args=eval_args,
            eval_dataset=tokenized_test,
            compute_metrics=compute_metrics,
        )

        eval_metrics = trainer.evaluate()
        self.local_validation_accuracy = eval_metrics["eval_accuracy"]
        self.local_validation_f1 = eval_metrics["eval_f1"]
        print(
            tag,
            f"Local model validation accuracy = {self.local_validation_accuracy:.4f}",
            f"F1 = {self.local_validation_f1:.4f}",
        )
        self.agg_validation_accuracy = eval_metrics["eval_accuracy"]

        self.next(self.join)

    @aggregator
    def join(self, inputs):
        """
        Model aggregation step.
        """
        tag = colored("[Aggregator]", "white", "on_magenta")
        print(tag, "Joining models from collaborators ...")

        # Average Training loss, aggregated and locally trained model accuracy
        sum_loss = 0.0
        sum_agg_acc = 0.0
        sum_agg_f1 = 0.0
        sum_loc_acc = 0.0
        sum_loc_f1 = 0.0
        n = len(inputs)

        for inp in inputs:
            sum_loss += inp.loss
            sum_agg_acc += inp.agg_validation_accuracy
            sum_agg_f1 += inp.agg_validation_f1
            sum_loc_acc += inp.local_validation_accuracy
            sum_loc_f1 += inp.local_validation_f1

        self.average_loss = sum_loss / n
        self.aggregated_model_accuracy = sum_agg_acc / n
        self.aggregated_model_f1 = sum_agg_f1 / n
        self.local_model_accuracy = sum_loc_acc / n
        self.local_model_f1 = sum_loc_f1 / n

        print(tag, f"Round {self.current_round}:")
        print(f"\tAvg. aggregated model validation accuracy = {self.aggregated_model_accuracy:.4f}")
        print(f"\tAvg. aggregated model validation f1 = {self.aggregated_model_f1:.4f}")
        print(f"\tAvg. training loss = {self.average_loss:.4f}")
        print(f"\tAvg. local model validation accuracy = {self.local_model_accuracy:.4f}")
        print(f"\tAvg. local model validation f1 = {self.local_model_f1:.4f}")

        # Averaging weights
        self.model = fed_avg(self.model, [inp.model for inp in inputs])

        self.results.append(
            [
                self.current_round,
                self.aggregated_model_accuracy,
                self.aggregated_model_f1,
                self.average_loss,
                self.local_model_accuracy,
                self.local_model_f1,
            ],
        )

        self.current_round += 1

        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation, foreach="collaborators")

        else:
            self.next(self.end)

    @aggregator
    def end(self):
        """
        This is the last step in the Flow.
        """
        tag = colored("[Aggregator]", "white", "on_magenta")
        print(tag, "This is the end of the flow")

### Simulation: LocalRuntime

We now import & define the `LocalRuntime`, participants (`Aggregator/Collaborator`), and initialize the private attributes for participants.

- `Runtime` – Defines where the flow runs. `LocalRuntime` simulates the flow on local node.
- `Aggregator/Collaborator` - (Local) Participants in the simulation

Since this cell is used for simulation, we don't use the export directive.

In [ ]:
from datasets import load_dataset

from openfl.experimental.workflow.interface import Aggregator, Collaborator
from openfl.experimental.workflow.runtime import LocalRuntime

# Setup Aggregator & initialize private attributes
agg = Aggregator()
agg.private_attributes = {}

# Setup Collaborators & initialize shards of MNIST dataset as private attributes
n_collaborators = 2
collaborator_names = ["Portland", "Seattle"]

# Load imdb dataset
imdb_dataset = load_dataset("imdb")

# Split dataset between collaborators
collaborators = [Collaborator(name=name) for name in collaborator_names]
for idx, collab in enumerate(collaborators):
    local_train = imdb_dataset["train"].select(
        list(
            range(idx, len(imdb_dataset["train"]), n_collaborators),
        ),
    )
    local_test = imdb_dataset["test"].select(
        list(
            range(idx, len(imdb_dataset["test"]), n_collaborators),
        ),
    )

    collab.private_attributes = {
        "train_dataset": local_train,
        "test_dataset": local_test,
    }

local_runtime = LocalRuntime(
    aggregator=agg,
    collaborators=collaborators,
    backend="single_process",
)
print(f"Local runtime collaborators = {local_runtime.collaborators}")

### Start Simulation

Now that we have our flow and runtime defined, let's run the simulation! 

In [ ]:
model = None
flflow = FederatedFlowHF(model, checkpoint=True)
flflow.runtime = local_runtime
flflow.run()

Let us check the simulation results

In [ ]:
from tabulate import tabulate

simulation_results = flflow.results
headers = [
    "Rounds",
    "Agg Model Validation Accuracy",
    "Agg Model Validation F1",
    "Local Train loss",
    "Local Model Validation Accuracy",
    "Local Model Validation F1",
]

print("********** Simulation results **********")
print(tabulate(simulation_results, headers=headers, tablefmt="outline"))

### Setup Federation: Director & Envoys

Before we can deploy the experiment, let us create participants in Federation: Director and Envoys. As the Tutorial uses two collaborators we shall launch three participants:
1. Director: The central node in the Federation
2. Portland: The first envoy in the Federation
3. Seattle: The second envoy in the Federation 

The participants can be launched by following steps mentioned in [README]((https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/workflow/FederatedRuntime/101_MNIST/README.md))


### Deploy: FederatedRuntime

We now import and instantiate `FederatedRuntime` to enable deployment of experiment on distributed infrastructure. Initializing the `FederatedRuntime` requires following inputs to be provided by the user:

- `director_info` – director information including fqdn of the director node, port, and certificate information
- `collaborators` - names of the collaborators participating in experiment
- `notebook_path`- path to this jupyter notebook


In [ ]:
# | export

from openfl.experimental.workflow.runtime import FederatedRuntime

director_info = {
    "director_node_fqdn": "localhost",
    "director_port": 50050,
}

federated_runtime = FederatedRuntime(
    collaborators=collaborator_names,
    director=director_info,
    notebook_path="./HF_FederatedRuntime.ipynb",
)

Let us connect to federation & check if the envoys are connected to the director by using the `get_envoys` method of `FederatedRuntime`. If the participants are launched successful in previous step the status of `Portland` and `Seattle` should be displayed as `Online`

In [ ]:
federated_runtime.get_envoys()

Now that we have our distributed infrastructure ready, let us modify the flow runtime to `FederatedRuntime` instance and deploy the experiment. 

Progress of the flow is available on 
1. Jupyter notebook: if `checkpoint` attribute of the flow object is set to `True`
2. Director and Envoy terminals  


In [ ]:
# Load imdb dataset
imdb_dataset = load_dataset("imdb")
collabs = ["Portland", "Seattle"]

for idx, c in enumerate(collabs):
    local_train = imdb_dataset["train"].select(list(range(idx, len(imdb_dataset["train"]), 2)))
    local_test = imdb_dataset["test"].select(list(range(idx, len(imdb_dataset["test"]), 2)))

    local_train.save_to_disk(f"../{c}/data/imdb_train_{c.lower()}")
    local_test.save_to_disk(f"../{c}/data/imdb_test_{c.lower()}")

In [ ]:
flflow.results = []  # clear results from previous run
flflow.runtime = federated_runtime
flflow.run()

Let us compare the simulation results from `LocalRuntime` and federation results from `FederatedRuntime`

In [ ]:
headers = [
    "Rounds",
    "Agg Model Validation Accuracy",
    "Agg Model Validation F1",
    "Local Train loss",
    "Local Model Validation Accuracy",
    "Local Model Validation F1",
]

print("********** Simulation results **********")
print(tabulate(simulation_results, headers=headers, tablefmt="outline"))

print("********** Federation results **********")
federation_results = flflow.results
print(tabulate(federation_results, headers=headers, tablefmt="outline"))

### Remove downloaded and generated files

In [ ]:
import shutil

for c in ["Portland", "Seattle"]:
    shutil.rmtree(f"../{c}/__pycache__")
    shutil.rmtree(f"../{c}/data/imdb_train_{c.lower()}")
    shutil.rmtree(f"../{c}/data/imdb_test_{c.lower()}")

shutil.rmtree("trainer_output")
shutil.rmtree("generated_workspace")